# `ml-from-scratch-core`: Interactive Tour & Verification

Welcome to **`ml-from-scratch-core`**, a zero-dependency (NumPy-only) reference implementation of foundational machine learning algorithms.

This notebook demonstrates:
1. **Logistic Regression with Adam**: Binary & Softmax classification with regularization.
2. **CART Decision Trees & Random Forests**: Gini / Entropy splitting and bagging.
3. **Gradient Boosted Decision Trees (GBDT)**: Friedman's functional gradient descent and residual fitting.
4. **Scalar Autograd Engine & MLP**: Micrograd-style reverse-mode automatic differentiation solving the non-linear XOR problem.
5. **Parity Benchmarks**: Direct comparison against `scikit-learn`.

In [1]:
import numpy as np
from sklearn.datasets import load_breast_cancer, load_iris, load_diabetes
from ml_from_scratch.linear import LogisticRegression
from ml_from_scratch.tree import DecisionTreeClassifier, RandomForestClassifier
from ml_from_scratch.ensemble import GradientBoostedClassifier, GradientBoostedRegressor
from ml_from_scratch.autograd import Value, MLP, mse_loss
from ml_from_scratch.metrics import accuracy_score, r2_score

## 1. Logistic Regression with Adam on Breast Cancer Dataset

In [2]:
data = load_breast_cancer()
X, y = data.data, data.target
X_norm = (X - X.mean(axis=0)) / (X.std(axis=0) + 1e-8)

clf = LogisticRegression(penalty='l2', C=1.0, optimizer='adam', learning_rate=0.05, max_iter=400, random_state=42)
clf.fit(X_norm, y)

acc = clf.score(X_norm, y)
print(f"Fitted Logistic Regression in {clf.n_iter_} iterations.")
print(f"Training Accuracy: {acc * 100:.2f}%")

## 2. Decision Tree & Random Forest on Iris Dataset

In [3]:
iris = load_iris()
X_iris, y_iris = iris.data, iris.target

dt = DecisionTreeClassifier(criterion='gini', max_depth=3, random_state=42)
dt.fit(X_iris, y_iris)
print(f"Decision Tree Accuracy: {dt.score(X_iris, y_iris) * 100:.2f}%")
print(f"Feature Importances: {dt.feature_importances_}")

rf = RandomForestClassifier(n_estimators=50, max_depth=4, oob_score=True, random_state=42)
rf.fit(X_iris, y_iris)
print(f"Random Forest Accuracy: {rf.score(X_iris, y_iris) * 100:.2f}%")
print(f"Out-of-Bag (OOB) Generalization Score: {rf.oob_score_ * 100:.2f}%")

## 3. Gradient Boosted Decision Tree (GBDT) on Breast Cancer

In [4]:
gbdt = GradientBoostedClassifier(n_estimators=40, learning_rate=0.1, max_depth=3, random_state=42)
gbdt.fit(X, y)
print(f"GBDT Classification Accuracy: {gbdt.score(X, y) * 100:.2f}%")

## 4. Scalar Autograd Engine & MLP Solving Non-Linear XOR

In [5]:
xs = [[2.0, 2.0], [2.0, -2.0], [-2.0, 2.0], [-2.0, -2.0]]
ys = [0.0, 1.0, 1.0, 0.0]

mlp = MLP(2, [4, 1], activations=['tanh', 'sigmoid'])

for epoch in range(120):
    preds = [mlp(x) for x in xs]
    loss = mse_loss(preds, ys)
    mlp.zero_grad()
    loss.backward()
    for p in mlp.parameters():
        p.data -= 0.5 * p.grad

print("Autograd MLP Predictions on XOR:")
for x, target in zip(xs, ys):
    pred_val = mlp(x).data
    print(f"  Input: {x} -> Predicted: {pred_val:.4f} (True: {target})")